# Phase 5: Validation

Leave-one-parent-out (LOPO) and leave-one-scenario-out (LOSO) cross-validation
for both Stage A (selection) and Stage B (sentiment).

Uses elastic-net models for CV (Bayesian refit per fold is too slow for N=20/45).
Metrics displayed in-notebook alongside baseline comparisons.

In [ ]:
import os, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              brier_score_loss, mean_absolute_error)
from sklearn.preprocessing import OneHotEncoder
import scipy.sparse as sp

SEED = 20260506
np.random.seed(SEED)

_nb_dir = os.path.abspath('.')
TRACES_DIR = os.path.join(_nb_dir, 'traces')

with open(os.path.join(TRACES_DIR, 'X_stage_a.pkl'), 'rb') as f:
    X_stage_a = pickle.load(f)
with open(os.path.join(TRACES_DIR, 'panel_meta.pkl'), 'rb') as f:
    panel_meta = pickle.load(f)
with open(os.path.join(TRACES_DIR, 'X_stage_b.pkl'), 'rb') as f:
    X_stage_b = pickle.load(f)
with open(os.path.join(TRACES_DIR, 'df_sel_aug.pkl'), 'rb') as f:
    df_sel_aug = pickle.load(f)
with open(os.path.join(TRACES_DIR, 'enet_coefs_a.pkl'), 'rb') as f:
    enet_a = pickle.load(f)

y_a = panel_meta['y'].values
y_b = df_sel_aug['highlight_sentiment'].astype(float).values

print(f'Stage A: {X_stage_a.shape}, positive rate={y_a.mean():.1%}')
print(f'Stage B: {X_stage_b.shape}, sentiment mean={y_b.mean():.2f}')

In [ ]:
# ── Shared CV helpers ─────────────────────────────────────────────────────

def build_enet_matrix(X, panel, enc_parent=None, enc_scenario=None, fit=True):
    """Add parent+scenario one-hot to feature matrix."""
    X_cont = sp.csr_matrix(X.fillna(0).values)
    if fit:
        enc_parent   = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
        enc_scenario = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
        p_ohe = enc_parent.fit_transform(panel[['parent_id']])
        s_ohe = enc_scenario.fit_transform(panel[['scenario_id']])
    else:
        p_ohe = enc_parent.transform(panel[['parent_id']])
        s_ohe = enc_scenario.transform(panel[['scenario_id']])
    return sp.hstack([X_cont, p_ohe, s_ohe]).toarray(), enc_parent, enc_scenario


def stage_a_metrics(y_true, y_prob):
    return {
        'pr_auc': average_precision_score(y_true, y_prob) if y_true.sum() > 0 else np.nan,
        'roc_auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': brier_score_loss(y_true, y_prob),
    }


def stage_b_metrics(y_true, y_pred):
    rho, _ = spearmanr(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    within_1 = np.mean(np.abs(y_true - y_pred) <= 1)
    return {'spearman_rho': rho, 'mae': mae, 'within_1': within_1}


BEST_C_A  = enet_a['best_C']
BEST_L1_A = enet_a['best_l1r']

print('Helpers defined.')

## 5.1  Leave-one-parent-out CV — Stage A

In [ ]:
parents = panel_meta['parent_id'].unique()
lopo_a_rows = []

for held_parent in parents:
    train_mask = panel_meta['parent_id'] != held_parent
    test_mask  = panel_meta['parent_id'] == held_parent

    X_tr, enc_p, enc_s = build_enet_matrix(X_stage_a[train_mask],
                                            panel_meta[train_mask], fit=True)
    X_te, _, _          = build_enet_matrix(X_stage_a[test_mask],
                                            panel_meta[test_mask],
                                            enc_parent=enc_p, enc_scenario=enc_s, fit=False)

    y_tr = y_a[train_mask]
    y_te = y_a[test_mask]

    clf = LogisticRegression(
        penalty='elasticnet', solver='saga',
        l1_ratio=BEST_L1_A, C=BEST_C_A,
        max_iter=500, random_state=SEED
    )
    clf.fit(X_tr, y_tr)
    prob = clf.predict_proba(X_te)[:, 1]

    m = stage_a_metrics(y_te, prob)
    m['parent_id'] = held_parent
    m['n_test'] = test_mask.sum()
    m['n_positive'] = y_te.sum()
    lopo_a_rows.append(m)

lopo_a_df = pd.DataFrame(lopo_a_rows)
print('LOPO Stage A metrics (per parent):')
display(lopo_a_df[['parent_id', 'n_test', 'n_positive', 'pr_auc', 'roc_auc', 'brier']].round(3))
print('\nAggregated:')
display(lopo_a_df[['pr_auc', 'roc_auc', 'brier']].describe().round(3))

## 5.2  Leave-one-scenario-out CV — Stage A

In [ ]:
scenarios = panel_meta['scenario_id'].unique()
loso_a_rows = []

for held_scen in scenarios:
    train_mask = panel_meta['scenario_id'] != held_scen
    test_mask  = panel_meta['scenario_id'] == held_scen

    if test_mask.sum() == 0:
        continue

    X_tr, enc_p, enc_s = build_enet_matrix(X_stage_a[train_mask],
                                            panel_meta[train_mask], fit=True)
    X_te, _, _          = build_enet_matrix(X_stage_a[test_mask],
                                            panel_meta[test_mask],
                                            enc_parent=enc_p, enc_scenario=enc_s, fit=False)

    y_tr = y_a[train_mask]
    y_te = y_a[test_mask]

    if len(np.unique(y_tr)) < 2:
        continue

    clf = LogisticRegression(
        penalty='elasticnet', solver='saga',
        l1_ratio=BEST_L1_A, C=BEST_C_A,
        max_iter=500, random_state=SEED
    )
    clf.fit(X_tr, y_tr)
    prob = clf.predict_proba(X_te)[:, 1]

    m = stage_a_metrics(y_te, prob)
    m['scenario_id'] = held_scen
    m['n_test'] = test_mask.sum()
    m['n_positive'] = y_te.sum()
    loso_a_rows.append(m)

loso_a_df = pd.DataFrame(loso_a_rows)
print('LOSO Stage A — aggregated metrics:')
display(loso_a_df[['pr_auc', 'roc_auc', 'brier']].describe().round(3))

## 5.3  Stage A baselines

In [ ]:
# Baseline 1: global mean positive rate
global_rate = y_a.mean()
baseline_global_a = stage_a_metrics(y_a, np.full(len(y_a), global_rate))

# Baseline 2: per-span empirical highlight rate (n_highlighters / n_parents)
with open(os.path.join(TRACES_DIR, 'span_universe_df.pkl'), 'rb') as f:
    span_universe_df = pickle.load(f)
n_parents = panel_meta['parent_id'].nunique()
span_rate = (span_universe_df.set_index('span_id')['n_highlighters'] / n_parents)
panel_meta_idx = panel_meta.set_index('span_id', drop=False)
per_span_prob = panel_meta['span_id'].map(span_rate).fillna(global_rate).values
baseline_perspan_a = stage_a_metrics(y_a, per_span_prob)

baseline_df_a = pd.DataFrame([
    {'model': 'Global mean rate (baseline)',      **baseline_global_a},
    {'model': 'Per-span empirical rate (baseline)', **baseline_perspan_a},
    {'model': 'Elastic net LOPO (mean)',
     **lopo_a_df[['pr_auc', 'roc_auc', 'brier']].mean().to_dict()},
    {'model': 'Elastic net LOSO (mean)',
     **loso_a_df[['pr_auc', 'roc_auc', 'brier']].mean().to_dict()},
])
print('Stage A — model vs baselines:')
display(baseline_df_a.round(4))

In [ ]:
# ── Calibration plot (pooled LOPO) ────────────────────────────────────────
# Collect held-out predictions from LOPO
all_probs_a, all_y_a = [], []
for held_parent in parents:
    train_mask = panel_meta['parent_id'] != held_parent
    test_mask  = panel_meta['parent_id'] == held_parent
    X_tr, enc_p, enc_s = build_enet_matrix(X_stage_a[train_mask], panel_meta[train_mask], fit=True)
    X_te, _, _ = build_enet_matrix(X_stage_a[test_mask], panel_meta[test_mask],
                                    enc_parent=enc_p, enc_scenario=enc_s, fit=False)
    clf = LogisticRegression(penalty='elasticnet', solver='saga',
                              l1_ratio=BEST_L1_A, C=BEST_C_A,
                              max_iter=500, random_state=SEED)
    clf.fit(X_tr, y_a[train_mask])
    all_probs_a.extend(clf.predict_proba(X_te)[:, 1])
    all_y_a.extend(y_a[test_mask])

all_probs_a = np.array(all_probs_a)
all_y_a = np.array(all_y_a)

bins = np.linspace(0, 1, 11)
bin_idx = np.digitize(all_probs_a, bins) - 1
bin_idx = np.clip(bin_idx, 0, 9)
mean_pred = [all_probs_a[bin_idx == i].mean() for i in range(10)]
mean_obs  = [all_y_a[bin_idx == i].mean()     for i in range(10)]

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Perfect calibration')
ax.plot(mean_pred, mean_obs, 'o-', color='steelblue', label='Model')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Stage A calibration (pooled LOPO)')
ax.legend()
plt.tight_layout()
plt.show()

## 5.4  Leave-one-parent-out CV — Stage B

In [ ]:
parents_b = df_sel_aug['parent_id'].unique()
lopo_b_rows = []

for held_parent in parents_b:
    train_mask = df_sel_aug['parent_id'] != held_parent
    test_mask  = df_sel_aug['parent_id'] == held_parent

    if test_mask.sum() < 2:
        continue

    X_tr_b = X_stage_b[train_mask].fillna(0).values
    X_te_b = X_stage_b[test_mask].fillna(0).values
    y_tr_b = y_b[train_mask]
    y_te_b = y_b[test_mask]

    # Ridge regression as ordinal proxy (fast, appropriate for 1–7 scale)
    reg = Ridge(alpha=1.0, random_state=SEED)
    reg.fit(X_tr_b, y_tr_b)
    pred = reg.predict(X_te_b)
    pred = np.clip(np.round(pred), 1, 7)

    m = stage_b_metrics(y_te_b, pred)
    m['parent_id'] = held_parent
    m['n_test'] = test_mask.sum()
    lopo_b_rows.append(m)

lopo_b_df = pd.DataFrame(lopo_b_rows)
print('LOPO Stage B metrics (per parent):')
display(lopo_b_df.round(3))
print('\nAggregated:')
display(lopo_b_df[['spearman_rho', 'mae', 'within_1']].describe().round(3))

## 5.5  Leave-one-scenario-out CV — Stage B

In [ ]:
scenarios_b = df_sel_aug['scenario_id'].unique()
loso_b_rows = []

for held_scen in scenarios_b:
    train_mask = df_sel_aug['scenario_id'] != held_scen
    test_mask  = df_sel_aug['scenario_id'] == held_scen

    if test_mask.sum() < 2:
        continue

    X_tr_b = X_stage_b[train_mask].fillna(0).values
    X_te_b = X_stage_b[test_mask].fillna(0).values
    y_tr_b = y_b[train_mask]
    y_te_b = y_b[test_mask]

    reg = Ridge(alpha=1.0, random_state=SEED)
    reg.fit(X_tr_b, y_tr_b)
    pred = np.clip(np.round(reg.predict(X_te_b)), 1, 7)

    m = stage_b_metrics(y_te_b, pred)
    m['scenario_id'] = held_scen
    m['n_test'] = test_mask.sum()
    loso_b_rows.append(m)

loso_b_df = pd.DataFrame(loso_b_rows)
print('LOSO Stage B — aggregated metrics:')
display(loso_b_df[['spearman_rho', 'mae', 'within_1']].describe().round(3))

## 5.6  Stage B baselines

In [ ]:
# Baseline 1: global mean sentiment
global_mean_b = y_b.mean()
baseline_global_b = stage_b_metrics(y_b, np.full(len(y_b), global_mean_b))

# Baseline 2: per-parent mean sentiment
parent_means = df_sel_aug.groupby('parent_id')['highlight_sentiment'].mean()
per_parent_pred = df_sel_aug['parent_id'].map(parent_means).fillna(global_mean_b).values
baseline_parent_b = stage_b_metrics(y_b, per_parent_pred)

baseline_df_b = pd.DataFrame([
    {'model': 'Global mean sentiment (baseline)',    **baseline_global_b},
    {'model': 'Per-parent mean sentiment (baseline)', **baseline_parent_b},
    {'model': 'Ridge LOPO (mean)',
     **lopo_b_df[['spearman_rho', 'mae', 'within_1']].mean().to_dict()},
    {'model': 'Ridge LOSO (mean)',
     **loso_b_df[['spearman_rho', 'mae', 'within_1']].mean().to_dict()},
])
print('Stage B — model vs baselines:')
display(baseline_df_b.round(4))

# Save validation results
with open(os.path.join(TRACES_DIR, 'validation_results.pkl'), 'wb') as f:
    pickle.dump({
        'lopo_a': lopo_a_df, 'loso_a': loso_a_df,
        'lopo_b': lopo_b_df, 'loso_b': loso_b_df,
        'baseline_a': baseline_df_a, 'baseline_b': baseline_df_b,
    }, f)
print('\nValidation results saved.')